# Start

Discover available data, inspect API specifications and save your first dataset.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the location

These three fields contain the selected tehsil when downloaded from GeoLibre. Edit them to explore another location, then restart the kernel and run from the top.


In [ ]:
state = "Bihar"
district = "Nalanda"
tehsil = "Hilsa"


## Set your API key

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and keys. This cell reuses `CORE_STACK_API_KEY` or asks privately and stores it in this kernel’s environment. The request header is `X-API-Key`.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", value.replace("(", "").replace(")", "")).strip("_").lower()
         for key, value in {"state": state, "district": district, "tehsil": tehsil}.items()}
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}


## Choose an API from the public specification

The [API specifications](https://api-doc.core-stack.org) describe each request. This cell lists all GET APIs and required parameters, then shows the selected API’s parameters and response fields as tables. Change `api_path` to inspect another API.


In [ ]:
response = requests.get("https://geoserver.core-stack.org/?format=openapi", timeout=90)
specification = read_json(response)
operations = {path: details["get"] for path, details in specification["paths"].items() if "get" in details and path.startswith("/get_")}
display(pd.DataFrame([{"path": path, "description": op.get("summary", ""),
                       "required_parameters": ", ".join(p["name"] for p in op.get("parameters", []) if p.get("required"))}
                      for path, op in operations.items()]))
api_path = "/get_active_locations/"
operation = operations[api_path]
display(pd.DataFrame(operation.get("parameters", [])).reindex(columns=["name", "required", "type", "description"]))
display(pd.json_normalize(operation.get("responses", {})).T)
references = re.findall(r'#/definitions/([^" ]+)', json.dumps(operation.get("responses", {})))
for name in dict.fromkeys(references):
    definition = specification.get("definitions", {}).get(name, {})
    display(pd.DataFrame(definition.get("properties", {})).T)


## Make the request and inspect its records

Use `{}` for active locations. For a tehsil API, set `parameters = place`. Add the identifier or coordinates required by the selected path. `record_path` selects a table inside a response, after it has been downloaded; it is not a server filter.


In [ ]:
parameters = {}
response = requests.get(API_URL + api_path.lstrip("/"), params=parameters, headers=api_headers, timeout=180)
api_result = read_json(response)
record_path = None  # For get_tehsil_data, try "mws"; for get_mws_data, "time_series".
records = api_result[record_path] if record_path else api_result
preview = pd.json_normalize(records)
display(preview.head())


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, "{state}/{district}/{tehsil}/collection.json".format(**place))
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
display(items)
dataset = "terrain_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
field_notes = pd.DataFrame(columns=["name", "type", "description"])
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(pd.DataFrame([item["properties"]]).reindex(columns=["title", "description", "start_datetime", "end_datetime"]).T)
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Discover available downloads

`get_generated_layer_urls` lists the published dataset downloads. Choose columns from this response to inspect styles or asset locations as well. STAC supplies their dataset and field descriptions.


In [ ]:
response = requests.get(API_URL + "get_generated_layer_urls/", params=place, headers=api_headers, timeout=180)
downloads = pd.DataFrame(read_json(response))
columns = ["dataset_name", "layer_type", "layer_url", "style_url"]
display(downloads.reindex(columns=columns))


## Read, inspect and save micro-watershed data

Join the boundary and attribute APIs on `uid`. The first record is shown as a table; the files can be opened in QGIS or GeoLibre. All source field names remain unchanged.


In [ ]:
response = requests.get(API_URL + "get_mws_geometries/", params=place, headers=api_headers, timeout=180)
mws = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
attributes = pd.DataFrame(api_data["mws"])
mws["uid"], attributes["uid"] = mws["uid"].astype(str), attributes["uid"].astype(str)
mws = mws.merge(attributes, on="uid", how="left", validate="one_to_one")
display(mws.drop(columns="geometry").iloc[0].to_frame("value"))
mws.to_file("micro_watersheds.geojson", driver="GeoJSON")
mws.drop(columns="geometry").to_csv("micro_watersheds.csv", index=False)
display(FileLink("micro_watersheds.geojson"), FileLink("micro_watersheds.csv"))


### Try another field

The tehsil response is already in `api_data`. Use `pd.DataFrame(api_data["terrain"])` or replace `"terrain"` with `"dem"`, `"hydrological_annual"`, `"croppingIntensity_annual"` or `"social_economic_indicator"`. Inspect `.columns`, then select just the columns you want. The next notebooks demonstrate these choices without downloading the same response again within a notebook.
